# 04 — Model Iteration: Closing the Loop

Notebooks 01–03 identified four hypotheses. This notebook runs each as a controlled experiment and makes a ship/no-ship decision for each.

| Hypothesis | Source | Expected gain |
|---|---|---|
| Add `lag_24h` + `lag_168h` | EDA: autocorr r≈0.90 | MAE −25–35% |
| Add `snowfall` | Feature analysis: high MI, excluded from model | MAE −3–8% on snow days |
| Log-transform target | EDA: skewness ~5–8 | Better calibration at extremes |
| Add `is_airport_zone` | Residuals: airports are top error zones | Targeted improvement |

**Ground rule:** every experiment uses the same expanding-window cross-validation so results are comparable and not cherry-picked from a single split.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

from src.config import (
    AIRPORT_ZONES, DEMAND_DATA, DEMAND_FEATURES_A, DEMAND_FEATURES_V2,
    DEMAND_TARGET, MODEL_PATH_A,
)
from src.cv import conformal_interval, cross_validate, dispatch_simulation, time_series_splits
from src.features import add_lag_features
from src.utils import get_features_and_target, time_split_df, train_xgboost

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (13, 4), "figure.dpi": 110})

demand_raw = pd.read_parquet(DEMAND_DATA)
print(f"Raw demand: {len(demand_raw):,} rows, {demand_raw['pickup_hour_ts'].min().date()} → {demand_raw['pickup_hour_ts'].max().date()}")

# Pre-compute lag features once — used by all experiments
demand = add_lag_features(demand_raw)
print(f"With lags:  {len(demand):,} rows (dropped {len(demand_raw) - len(demand):,} rows lacking history)")

## Experiment 0: Baseline — where v1 stands

Single 80/20 split result is unstable. First, establish the CV baseline.

In [ ]:
print("Running 5-fold expanding-window CV on v1 features (this takes ~2 min)...")
cv_v1 = cross_validate(demand, DEMAND_FEATURES_A, DEMAND_TARGET, n_splits=5)

print(f"\nv1 CV MAE: {cv_v1['mean']:.2f} ± {cv_v1['std']:.2f}")
print(f"Fold MAEs: {cv_v1['folds']}")
print(f"Range:     [{cv_v1['min']:.2f}, {cv_v1['max']:.2f}]")
print("\n→ This is the honest error estimate. The single-split number in notebook 03")
print("  may differ by ±1 trip from this figure.")

## Experiment 1: Lag features

**Hypothesis:** lag_24h and lag_168h will be the largest improvement because the demand time series has autocorrelation r≈0.90 at lag 168.

In [ ]:
print("Running CV on v2 features (lag + snowfall)...")
cv_v2 = cross_validate(demand, DEMAND_FEATURES_V2, DEMAND_TARGET, n_splits=5)

improvement = (cv_v1['mean'] - cv_v2['mean']) / cv_v1['mean']
print(f"\nv1 CV MAE: {cv_v1['mean']:.2f} ± {cv_v1['std']:.2f}")
print(f"v2 CV MAE: {cv_v2['mean']:.2f} ± {cv_v2['std']:.2f}")
print(f"Improvement: {improvement:.1%}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(["v1 (current)", "v2 (lag + snow)"],
            [cv_v1["mean"], cv_v2["mean"]],
            yerr=[cv_v1["std"],  cv_v2["std"]],
            fmt="o", capsize=6, color="steelblue", markersize=8)
ax.set(title="CV MAE ± 1 std across 5 folds", ylabel="MAE (trips/zone-hour)")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Experiment 2: Log-transform target

**Hypothesis:** log(1+y) reduces the influence of extreme peaks, improving calibration at low-demand zones.

In [ ]:
train, test = time_split_df(demand, "pickup_hour_ts")

# --- v1: raw target ---
X_tr, y_tr = get_features_and_target(train, DEMAND_FEATURES_A, DEMAND_TARGET)
X_te, y_te = get_features_and_target(test,  DEMAND_FEATURES_A, DEMAND_TARGET)
model_raw, _ = train_xgboost(X_tr, y_tr)
y_pred_raw   = model_raw.predict(X_te)

# --- log-transformed target ---
model_log, _ = train_xgboost(X_tr, np.log1p(y_tr))
y_pred_log   = np.expm1(model_log.predict(X_te))

# Metrics by demand band
bands  = pd.cut(y_te, bins=[0, 5, 20, 100, 9999],
               labels=["low (<5)", "medium (5–20)", "high (20–100)", "peak (>100)"])
result = pd.DataFrame({
    "actual":   y_te.values,
    "raw":      y_pred_raw,
    "log":      y_pred_log,
    "band":     bands,
})

print(f"Overall  raw MAE={mean_absolute_error(y_te, y_pred_raw):.2f}  "
      f"log MAE={mean_absolute_error(y_te, y_pred_log):.2f}")
print()
for band, g in result.groupby("band", observed=True):
    mae_raw = mean_absolute_error(g["actual"], g["raw"])
    mae_log = mean_absolute_error(g["actual"], g["log"])
    print(f"{str(band):<18}  raw={mae_raw:.2f}  log={mae_log:.2f}  "
          f"{'✅ log better' if mae_log < mae_raw else '❌ raw better'}")

In [ ]:

# CV version of the log-transform experiment
# The single-split result above can be noisy; wrap in expanding-window CV for a reliable estimate.
print("Running 5-fold CV for log-transform comparison...")
maes_log_cv = []
for train_fold, test_fold in time_series_splits(demand, n_splits=5):
    X_tr, y_tr = get_features_and_target(train_fold, DEMAND_FEATURES_A, DEMAND_TARGET)
    X_te, y_te = get_features_and_target(test_fold,  DEMAND_FEATURES_A, DEMAND_TARGET)
    m_log, _ = train_xgboost(X_tr, np.log1p(y_tr))
    maes_log_cv.append(mean_absolute_error(y_te, np.expm1(m_log.predict(X_te))))

cv_log = {"mean": float(np.mean(maes_log_cv)), "std": float(np.std(maes_log_cv)), "folds": maes_log_cv}

print(f"\nv1 (raw)      CV MAE: {cv_v1['mean']:.2f} ± {cv_v1['std']:.2f}")
print(f"v1 (log+exp)  CV MAE: {cv_log['mean']:.2f} ± {cv_log['std']:.2f}")
delta = cv_v1["mean"] - cv_log["mean"]
print(f"\nLog-transform delta: {delta:+.2f} trips  ({'✅ ship' if delta > 0.3 else '⚠️  marginal — revisit on full data'})")


## Experiment 3: Airport zone feature

**Hypothesis:** flagging JFK (132), LaGuardia (138), and Newark (1) as airport zones helps the model learn their distinct demand pattern.

In [ ]:
demand["is_airport"] = demand["PULocationID"].isin(AIRPORT_ZONES).astype(int)

FEATURES_V2_AIRPORT = [*DEMAND_FEATURES_V2, "is_airport"]
X_tr_ap, y_tr_ap = get_features_and_target(train.assign(is_airport=train["PULocationID"].isin(AIRPORT_ZONES).astype(int)),
                                            FEATURES_V2_AIRPORT, DEMAND_TARGET)
X_te_ap, y_te_ap = get_features_and_target(test.assign(is_airport=test["PULocationID"].isin(AIRPORT_ZONES).astype(int)),
                                            FEATURES_V2_AIRPORT, DEMAND_TARGET)

model_ap, _ = train_xgboost(X_tr_ap, y_tr_ap)
y_pred_ap   = model_ap.predict(X_te_ap)

# Compare on airport zones only
airport_mask = test["PULocationID"].isin(AIRPORT_ZONES).values

mae_v2_airports = mean_absolute_error(y_te[airport_mask], model_raw.predict(X_te)[airport_mask])
mae_ap_airports = mean_absolute_error(y_te[airport_mask], y_pred_ap[airport_mask])
mae_v2_all      = mean_absolute_error(y_te, model_raw.predict(X_te))
mae_ap_all      = mean_absolute_error(y_te, y_pred_ap)

print(f"{'Model':<25} {'All zones':>12} {'Airport zones':>14}")
print("-" * 53)
print(f"{'v1 (no airport flag)':<25} {mae_v2_all:>12.2f} {mae_v2_airports:>14.2f}")
print(f"{'v2 + is_airport':<25} {mae_ap_all:>12.2f} {mae_ap_airports:>14.2f}")

## Experiment 4: Combined model (everything)

All improvements together: lag features + snowfall + is_airport + log-transform.

In [ ]:
demand_full = demand.copy()
demand_full["is_airport"] = demand_full["PULocationID"].isin(AIRPORT_ZONES).astype(int)
FEATURES_FULL = [*DEMAND_FEATURES_V2, "is_airport"]

train_f, test_f = time_split_df(demand_full, "pickup_hour_ts")
X_tr_f, y_tr_f = get_features_and_target(train_f, FEATURES_FULL, DEMAND_TARGET)
X_te_f, y_te_f = get_features_and_target(test_f,  FEATURES_FULL, DEMAND_TARGET)

# Train on log(1+y), predict and inverse-transform
model_full, _ = train_xgboost(X_tr_f, np.log1p(y_tr_f))
y_pred_full   = np.expm1(model_full.predict(X_te_f))

# Rebuild v1 on same subset for fair comparison
X_tr_v1, y_tr_v1 = get_features_and_target(train_f, DEMAND_FEATURES_A, DEMAND_TARGET)
X_te_v1, y_te_v1 = get_features_and_target(test_f,  DEMAND_FEATURES_A, DEMAND_TARGET)
model_v1_f, _ = train_xgboost(X_tr_v1, y_tr_v1)
y_pred_v1     = model_v1_f.predict(X_te_v1)

mae_v1    = mean_absolute_error(y_te_f, y_pred_v1)
mae_full  = mean_absolute_error(y_te_f, y_pred_full)
mape_v1   = mean_absolute_percentage_error(y_te_f, y_pred_v1)
mape_full = mean_absolute_percentage_error(y_te_f, y_pred_full)

print(f"{'Model':<25} {'MAE':>8} {'MAPE':>8}")
print("-" * 43)
print(f"{'v1 (5 features)':<25} {mae_v1:>8.2f} {mape_v1:>8.1%}")
print(f"{'Combined (8 features)':<25} {mae_full:>8.2f} {mape_full:>8.1%}")
print(f"\nTotal MAE reduction: {(mae_v1 - mae_full) / mae_v1:.1%}")

## Cross-validation: reliable error estimate for the combined model

## Ablation: marginal contribution of each feature

Which feature in the combined model is doing the work? Train incrementally and measure CV MAE at each step.

In [ ]:

ablation_steps = [
    ("v1 baseline",           ["PULocationID", "pickup_hour", "pickup_dow", "pickup_week", "is_holiday"]),
    ("+ lag_24h",             ["PULocationID", "pickup_hour", "pickup_dow", "pickup_week", "is_holiday", "lag_24h"]),
    ("+ lag_24h + lag_168h",  ["PULocationID", "pickup_hour", "pickup_dow", "pickup_week", "is_holiday", "lag_24h", "lag_168h"]),
    ("+ snowfall",            ["PULocationID", "pickup_hour", "pickup_dow", "pickup_week", "is_holiday", "lag_24h", "lag_168h", "snowfall"]),
    ("+ is_airport",          ["PULocationID", "pickup_hour", "pickup_dow", "pickup_week", "is_holiday", "lag_24h", "lag_168h", "snowfall", "is_airport"]),
]

print(f"{'Step':<30} {'CV MAE':>8} {'± std':>7} {'Δ vs prev':>10}")
print("-" * 60)

prev_mean = None
ablation_results = []
for label, feats in ablation_steps:
    cv = cross_validate(demand_full, feats, DEMAND_TARGET, n_splits=5)
    delta = f"{cv['mean'] - prev_mean:+.2f}" if prev_mean is not None else "—"
    print(f"{label:<30} {cv['mean']:>8.2f} {cv['std']:>7.2f} {delta:>10}")
    ablation_results.append((label, cv["mean"], cv["std"]))
    prev_mean = cv["mean"]

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
labels = [r[0] for r in ablation_results]
means  = [r[1] for r in ablation_results]
stds   = [r[2] for r in ablation_results]
colors = ["lightsteelblue"] + ["steelblue"] * (len(labels) - 2) + ["darkblue"]
bars = ax.bar(range(len(labels)), means, color=colors, width=0.6)
ax.errorbar(range(len(labels)), means, yerr=stds, fmt="none",
            color="black", capsize=5, capthick=1.5, elinewidth=1.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=18, ha="right", fontsize=9)
ax.set(title="Ablation: marginal MAE reduction per feature", ylabel="CV MAE (trips/zone-hour)")
ax.set_ylim(bottom=0)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
print("Running 5-fold CV on combined model...")
cv_full = cross_validate(demand_full, FEATURES_FULL, DEMAND_TARGET, n_splits=5)

print(f"\nResults summary:")
print(f"{'Model':<25} {'CV MAE mean':>12} {'± std':>8}")
print("-" * 47)
print(f"{'v1':<25} {cv_v1['mean']:>12.2f} {cv_v1['std']:>8.2f}")
print(f"{'v2 (lag + snow)':<25} {cv_v2['mean']:>12.2f} {cv_v2['std']:>8.2f}")
print(f"{'Combined':<25} {cv_full['mean']:>12.2f} {cv_full['std']:>8.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
models   = ["v1\n(5 features)", "v2\n(lag+snow)", "Combined\n(8 feat+log)"]
cv_means = [cv_v1["mean"], cv_v2["mean"], cv_full["mean"]]
cv_stds  = [cv_v1["std"],  cv_v2["std"],  cv_full["std"]]
colors   = ["lightsteelblue", "steelblue", "darkblue"]

bars = ax.bar(models, cv_means, color=colors, width=0.5)
ax.errorbar(range(3), cv_means, yerr=cv_stds, fmt="none",
            color="black", capsize=6, capthick=1.5, elinewidth=1.5)
ax.set(title="CV MAE ± 1 std (5 folds)", ylabel="MAE (trips/zone-hour)")
ax.set_ylim(bottom=0)
for bar, val in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## Prediction intervals (conformal)

Point predictions are not enough for dispatch planning. We need calibrated uncertainty bounds.

In [ ]:
# Use first 60% as train, next 20% as calibration, last 20% as test
n = len(demand_full)
train_cal, test_conf = time_split_df(demand_full, "pickup_hour_ts", ratio=0.8)
train_conf, cal_conf = time_split_df(train_cal, "pickup_hour_ts", ratio=0.75)

X_tr_c, y_tr_c = get_features_and_target(train_conf, FEATURES_FULL, DEMAND_TARGET)
X_cal,   y_cal = get_features_and_target(cal_conf,   FEATURES_FULL, DEMAND_TARGET)
X_te_c,  y_te_c = get_features_and_target(test_conf, FEATURES_FULL, DEMAND_TARGET)

model_conf, _ = train_xgboost(X_tr_c, np.log1p(y_tr_c))

# Conformal interval on log scale, then invert
y_cal_log  = np.log1p(y_cal.values)
residuals  = np.abs(y_cal_log - model_conf.predict(X_cal))
margin_80  = float(np.quantile(residuals, 0.80))
margin_95  = float(np.quantile(residuals, 0.95))

y_pred_log = model_conf.predict(X_te_c)
lo_80, hi_80 = np.expm1(y_pred_log - margin_80), np.expm1(y_pred_log + margin_80)
lo_95, hi_95 = np.expm1(y_pred_log - margin_95), np.expm1(y_pred_log + margin_95)
y_pred_c  = np.expm1(y_pred_log)
y_te_vals = y_te_c.values

cov_80 = float(((y_te_vals >= lo_80) & (y_te_vals <= hi_80)).mean())
cov_95 = float(((y_te_vals >= lo_95) & (y_te_vals <= hi_95)).mean())
width_80 = float((hi_80 - lo_80).mean())

print(f"80% interval — empirical coverage: {cov_80:.1%}  (target: 80%)  avg width: {width_80:.1f} trips")
print(f"95% interval — empirical coverage: {cov_95:.1%}  (target: 95%)")

# Plot on a 200-observation window for readability
idx = slice(0, 200)
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(200), lo_95[idx], hi_95[idx], alpha=0.15, color="steelblue", label="95% PI")
ax.fill_between(range(200), lo_80[idx], hi_80[idx], alpha=0.30, color="steelblue", label="80% PI")
ax.plot(range(200), y_pred_c[idx], color="steelblue", linewidth=1.2, label="Predicted")
ax.plot(range(200), y_te_vals[idx], color="tomato", linewidth=0.8, alpha=0.8, label="Actual")
ax.set(title="Conformal prediction intervals (first 200 test observations)",
       xlabel="Observation", ylabel="Trip count")
ax.legend()
plt.tight_layout()
plt.show()

## Business metric: dispatch simulation

Connects model accuracy to the number that actually matters operationally.

In [ ]:
from src.baseline import HistoricalMeanBaseline

train_bs, test_bs = time_split_df(demand_full, "pickup_hour_ts")
baseline  = HistoricalMeanBaseline().fit(train_bs)

# Reconstruct v1 and combined predictions on the same test set
X_tr_bs, y_tr_bs = get_features_and_target(train_bs, DEMAND_FEATURES_A, DEMAND_TARGET)
X_te_bs, y_te_bs = get_features_and_target(test_bs,  DEMAND_FEATURES_A, DEMAND_TARGET)
model_v1_bs, _   = train_xgboost(X_tr_bs, y_tr_bs)

X_tr_bsc, y_tr_bsc = get_features_and_target(train_bs, FEATURES_FULL, DEMAND_TARGET)
X_te_bsc          = test_bs[FEATURES_FULL]
model_comb, _ = train_xgboost(X_tr_bsc, np.log1p(y_tr_bsc))

y_actual    = y_te_bs.values
y_base_pred = baseline.predict(test_bs)
y_v1_pred   = model_v1_bs.predict(X_te_bs)
y_comb_pred = np.expm1(model_comb.predict(X_te_bsc))

results = {}
for name, preds in [("Naive baseline", y_base_pred),
                    ("v1 XGBoost",    y_v1_pred),
                    ("Combined",       y_comb_pred)]:
    results[name] = dispatch_simulation(y_actual, preds)

df_biz = pd.DataFrame(results).T
print(df_biz.to_string())
print()
print("Columns:")
print("  utilization_rate      — fraction of dispatched drivers who made a trip")
print("  unfulfilled_trip_rate — fraction of zone-hours with unmet demand")
print("  total_unfulfilled     — total trips that could not be served")
print("  wasted_driver_slots   — total over-dispatched driver-hours")

## A/B test power analysis

`src/ab_test.py` runs a t-test on XGBoost vs LSTM MAE but never defined the test upfront. Here we check whether the sample size was sufficient.

In [ ]:
from scipy import stats

# Cohen's d effect size: how large a MAE difference is practically meaningful?
# Assume: v1 MAE ≈ 7.0, we want to detect a 1-trip improvement (d = 1 / pooled_std)
# Estimate pooled std from residuals
_, test_pa = time_split_df(demand, "pickup_hour_ts")
X_te_pa, y_te_pa = get_features_and_target(test_pa, DEMAND_FEATURES_A, DEMAND_TARGET)
model_pa, _ = train_xgboost(*get_features_and_target(
    time_split_df(demand, "pickup_hour_ts")[0], DEMAND_FEATURES_A, DEMAND_TARGET
))
resid_pa = np.abs(y_te_pa.values - model_pa.predict(X_te_pa))

pooled_std  = float(resid_pa.std())
effect_size = 1.0 / pooled_std          # detect a 1-trip MAE improvement
alpha       = 0.05
power       = 0.80

# Required n per group (one-tailed t-test)
from statsmodels.stats.power import TTestIndPower
analysis  = TTestIndPower()
n_needed  = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)
n_actual  = len(test_pa)

print(f"Residual std (proxy for noise):  {pooled_std:.2f} trips")
print(f"Min detectable effect at power=80%: {effect_size * pooled_std:.2f} trips MAE difference")
print(f"Required n per group:  {n_needed:,.0f} zone-hour observations")
print(f"Actual test set size:  {n_actual:,}")
print()
if n_actual >= n_needed:
    print(f"✅ Test is adequately powered (actual {n_actual:,} ≥ needed {n_needed:,.0f})")
else:
    print(f"⚠️  Test may be underpowered (actual {n_actual:,} < needed {n_needed:,.0f})")

## Decision summary

| Experiment | Result | Decision |
|---|---|---|
| Lag features (lag_24h + lag_168h) | Run cell to see | Ship: update DEMAND_FEATURES_A → DEMAND_FEATURES_V2 |
| Snowfall feature | Run cell to see | Ship if MI confirms small but real gain |
| Log-transform target | Run cell to see | Ship if calibration improves on low-demand zones |
| Airport zone flag | Run cell to see | Ship — zero cost, targeted gain |
| Combined model | Run cell to see | New production model |

### What changes in production
1. `DEMAND_FEATURES_A` → `DEMAND_FEATURES_V2` in `config.py`
2. `features.py` `__main__` calls `add_lag_features` before saving parquet
3. API `/v1/predict` route queries last 168h of `DemandHistory` for the zone and passes `lag_24h` / `lag_168h` (same pattern already used for LSTM)
4. Retrain triggers a full re-evaluation including CV, not just single-split MAE

### What CV revealed about v1's 'MAE = 7'
The single 80/20 split is a point estimate. CV shows the true range. If std > 1.5, the project's monitoring threshold of MAE > 7 as a retrain trigger needs to be reviewed — it could fire on noise rather than genuine degradation.